In [1]:
import pandas as pd
import kagglehub
import os
from pathlib import Path
from plotly import express as px

c:\Users\felip\OneDrive\git_work\gaming-and-mental-health-analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.makedirs("../data/processed", exist_ok=True)

In [3]:
os.makedirs("../data/raw", exist_ok=True)

In [4]:
# Download latest version
origem = kagglehub.dataset_download("dreamtensor/gaming-addiction-and-mental-health-analysis")

print("Path to dataset files:", origem)

Path to dataset files: C:\Users\felip\.cache\kagglehub\datasets\dreamtensor\gaming-addiction-and-mental-health-analysis\versions\1


In [5]:
origem = Path(origem)
destino = Path('../data/raw')
destino.mkdir(parents=True, exist_ok=True)

for arquivo in origem.iterdir():
   if arquivo.is_file():
      arquivo.replace(destino / arquivo.name)

In [6]:
df = pd.read_csv('../data/raw/gaming_addiction.csv')

In [7]:
df

,user_id,age,gender,country,occupation,income_level,years_gaming,preferred_genre,platform,device_type,...,absenteeism_days,internet_speed_mbps,screen_time_total_hours,behavioral_cluster,addiction_score,addiction_binary,addiction_severity,burnout_probability,mental_health_risk_score,churn_probability
0,USR000001,21,Male,India,Employed,Middle,9,Sandbox,PC,Laptop,...,7,39.9,5.4,Casual Enjoyer,27.61,0,Mild,1.0,0.920,1.000
1,USR000002,25,Male,India,Employed,Lower-Middle,13,RPG,Mobile,Mixed,...,6,71.5,13.4,Streamer/Creator,55.51,1,Moderate,1.0,0.515,0.813
2,USR000003,26,Male,Brazil,Employed,Middle,14,RPG,PC+Mobile,High-end PC,...,7,119.4,12.3,Streamer/Creator,45.85,0,Moderate,1.0,0.720,0.947
3,USR000004,22,Male,South Korea,Employed,Upper-Middle,10,Strategy,PC+Mobile,Mobile,...,8,136.5,6.9,Toxic Competitor,39.87,0,Mild,1.0,0.520,0.660
4,USR000005,17,Female,India,Student,Middle,5,Strategy,PC,Laptop,...,6,78.8,9.3,Competitive Grinder,46.97,0,Moderate,1.0,0.585,0.867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,USR000246,23,Female,Japan,Streamer/Content Creator,Upper-Middle,11,MMORPG,PC+Console,Console,...,4,180.9,7.2,Streamer/Creator,42.37,0,Moderate,1.0,0.395,0.590
246,USR000247,18,Male,India,Student,Low,6,Sandbox,PC,Mid-range PC,...,0,91.5,5.6,Social Gamer,22.45,0,Mild,1.0,0.715,0.803
247,USR000248,20,Male,USA,Student,Middle,8,MMORPG,Mobile,Console,...,8,120.6,4.9,Escape Seeker,34.76,0,Mild,1.0,0.410,0.939
248,USR000249,19,Male,South Korea,Employed,Middle,7,Sandbox,PC+Console,High-end PC,...,2,175.9,5.2,Social Gamer,36.60,0,Mild,1.0,0.405,0.870


In [8]:
df.isna().sum().sum()

np.int64(119)

In [9]:
df.columns

Index(['user_id', 'age', 'gender', 'country', 'occupation', 'income_level',
       'years_gaming', 'preferred_genre', 'platform', 'device_type',
       'rank_tier', 'daily_playtime_hours', 'weekly_play_sessions',
       'late_night_sessions_hours', 'weekend_playtime_hours',
       'consecutive_hours_max', 'multiplayer_ratio', 'toxic_chat_reports',
       'rage_quit_frequency', 'in_game_purchases', 'monthly_spending_usd',
       'lootbox_openings', 'subscription_status', 'stress_score',
       'loneliness_score', 'dopamine_dependency_index', 'self_control_score',
       'impulsiveness_score', 'anxiety_level', 'depression_indicator',
       'emotional_stability', 'sleep_hours', 'exercise_frequency_per_week',
       'caffeine_intake_cups_day', 'social_interaction_hours',
       'relationship_status', 'gpa_or_performance_score', 'missed_deadlines',
       'productivity_drop_percent', 'absenteeism_days', 'internet_speed_mbps',
       'screen_time_total_hours', 'behavioral_cluster', 'addicti

In [10]:
# Distribuição depression_indicator
px.histogram(df, x='depression_indicator')

In [11]:
px.histogram(df, x='subscription_status')

In [12]:
px.histogram(df, x='gpa_or_performance_score', nbins=50)

In [13]:
px.histogram(df,x='addiction_severity')

In [14]:
df['subscription_status'] = df['subscription_status'].fillna('No subscription')

df['addiction_severity'] = df['addiction_severity'].fillna('Witout addiction')

df['depression_indicator'] = pd.to_numeric(
   df['depression_indicator'].replace('', pd.NA),
   errors='coerce'
)

df['gpa_or_performance_score'] = pd.to_numeric(
   df['gpa_or_performance_score'].replace('', pd.NA),
   errors='coerce'
)

df['depression_indicator'] = df['depression_indicator'].fillna(
   df.groupby('addiction_severity')['depression_indicator'].transform('median')
).fillna(4.5)

df['gpa_or_performance_score'] = df['gpa_or_performance_score'].fillna(
   df.groupby('occupation')['gpa_or_performance_score'].transform('median')
).fillna(4.0)

In [15]:
traducoes_colunas = {
    'user_id': 'id_usuario',
    'age': 'idade',
    'gender': 'genero',
    'country': 'pais',
    'occupation': 'ocupacao',
    'income_level': 'nivel_renda',
    'years_gaming': 'anos_jogando',
    'preferred_genre': 'genero_preferido',
    'platform': 'plataforma',
    'device_type': 'tipo_dispositivo',
    'rank_tier': 'nivel_ranqueado',
    'daily_playtime_hours': 'horas_jogo_diarias',
    'weekly_play_sessions': 'sessoes_semanais',
    'late_night_sessions_hours': 'horas_madrugada',
    'weekend_playtime_hours': 'horas_fim_de_semana',
    'consecutive_hours_max': 'max_horas_consecutivas',
    'multiplayer_ratio': 'proporcao_multiplayer',
    'toxic_chat_reports': 'denuncias_chat_toxico',
    'rage_quit_frequency': 'frequencia_rage_quit',
    'in_game_purchases': 'compras_no_jogo',
    'monthly_spending_usd': 'gasto_mensal_usd',
    'lootbox_openings': 'aberturas_lootbox',
    'subscription_status': 'status_assinatura',
    'stress_score': 'score_estresse',
    'loneliness_score': 'score_solidao',
    'dopamine_dependency_index': 'indice_dependencia_dopamina',
    'self_control_score': 'score_autocontrole',
    'impulsiveness_score': 'score_impulsividade',
    'anxiety_level': 'nivel_ansiedade',
    'depression_indicator': 'indicador_depressao',
    'emotional_stability': 'estabilidade_emocional',
    'sleep_hours': 'horas_sono',
    'exercise_frequency_per_week': 'exercicios_por_semana',
    'caffeine_intake_cups_day': 'xicaras_cafeina_dia',
    'social_interaction_hours': 'horas_interacao_social',
    'relationship_status': 'status_relacionamento',
    'gpa_or_performance_score': 'nota_ou_performance',
    'missed_deadlines': 'prazos_perdidos',
    'productivity_drop_percent': 'queda_produtividade_percentual',
    'absenteeism_days': 'dias_absenteismo',
    'internet_speed_mbps': 'velocidade_internet_mbps',
    'screen_time_total_hours': 'horas_tela_total',
    'behavioral_cluster': 'cluster_comportamental',
    'addiction_score': 'score_dependencia',
    'addiction_binary': 'dependencia_binaria',
    'addiction_severity': 'severidade_dependencia',
    'burnout_probability': 'probabilidade_burnout',
    'mental_health_risk_score': 'score_risco_saude_mental',
    'churn_probability': 'probabilidade_churn',
    'addiction_severity_dash': 'severidade_dependencia_dashboard'
}
df_powerbi = df.copy()
df_powerbi = df_powerbi.rename(columns=traducoes_colunas)

df_powerbi.head()

,id_usuario,idade,genero,pais,ocupacao,nivel_renda,anos_jogando,genero_preferido,plataforma,tipo_dispositivo,...,dias_absenteismo,velocidade_internet_mbps,horas_tela_total,cluster_comportamental,score_dependencia,dependencia_binaria,severidade_dependencia,probabilidade_burnout,score_risco_saude_mental,probabilidade_churn
0,USR000001,21,Male,India,Employed,Middle,9,Sandbox,PC,Laptop,...,7,39.9,5.4,Casual Enjoyer,27.61,0,Mild,1.0,0.920,1.000
1,USR000002,25,Male,India,Employed,Lower-Middle,13,RPG,Mobile,Mixed,...,6,71.5,13.4,Streamer/Creator,55.51,1,Moderate,1.0,0.515,0.813
2,USR000003,26,Male,Brazil,Employed,Middle,14,RPG,PC+Mobile,High-end PC,...,7,119.4,12.3,Streamer/Creator,45.85,0,Moderate,1.0,0.720,0.947
3,USR000004,22,Male,South Korea,Employed,Upper-Middle,10,Strategy,PC+Mobile,Mobile,...,8,136.5,6.9,Toxic Competitor,39.87,0,Mild,1.0,0.520,0.660
4,USR000005,17,Female,India,Student,Middle,5,Strategy,PC,Laptop,...,6,78.8,9.3,Competitive Grinder,46.97,0,Moderate,1.0,0.585,0.867


In [16]:
traducoes_valores = {
    'genero': {
        'Male': 'Masculino',
        'Female': 'Feminino',
        'Non-binary': 'Nao binario',
        'Prefer not to say': 'Prefiro nao informar'
    },
    'pais': {
        'Australia': 'Austrália',
        'Brazil': 'Brasil',
        'Canada': 'Canadá',
        'China': 'China',
        'France': 'França',
        'Germany': 'Alemanha',
        'India': 'Índia',
        'Indonesia': 'Indonésia',
        'Japan': 'Japão',
        'Mexico': 'México',
        'Other': 'Outro',
        'Russia': 'Rússia',
        'South Korea': 'Coréia do Sul',
        'UK': 'Reino Unido',
        'USA': 'Estados Unidos'
    },
    'ocupacao': {
        'Employed': 'Empregado',
        'Freelancer': 'Freelancer',
        'Streamer/Content Creator': 'Streamer/Criador de conteudo',
        'Student': 'Estudante',
        'Unemployed': 'Desempregado'
    },
    'nivel_renda': {
        'Low': 'Baixa',
        'Lower-Middle': 'Média-baixa',
        'Middle': 'Média',
        'Upper-Middle': 'Média-alta',
        'High': 'Alta'
    },
    'genero_preferido': {
        'Battle Royale': 'Battle Royale',
        'Casual': 'Casual',
        'FPS': 'FPS',
        'Horror': 'Terror',
        'MMORPG': 'MMORPG',
        'MOBA': 'MOBA',
        'RPG': 'RPG',
        'Sandbox': 'Sandbox',
        'Sports': 'Esportes',
        'Strategy': 'Estratégia'
    },
    'plataforma': {
        'Console': 'Console',
        'Mobile': 'Mobile',
        'PC': 'PC',
        'PC+Console': 'PC e console',
        'PC+Mobile': 'PC e mobile'
    },
    'tipo_dispositivo': {
        'Console': 'Console',
        'High-end PC': 'PC high-end',
        'Laptop': 'Notebook',
        'Mid-range PC': 'PC intermediario',
        'Mixed': 'Misto',
        'Mobile': 'Mobile'
    },
    'nivel_ranqueado': {
        'Bronze': 'Bronze',
        'Silver': 'Prata',
        'Gold': 'Ouro',
        'Platinum': 'Platina',
        'Diamond': 'Diamante',
        'Master': 'Mestre',
        'Grandmaster': 'Grao-mestre',
        'Unranked': 'Sem ranking'
    },
    'status_assinatura': {
        'None': 'Sem assinatura',
        'Basic': 'Básica',
        'Premium': 'Premium',
        'Ultimate': 'Ultimate'
    },
    'status_relacionamento': {
        'Divorced': 'Divorciado',
        'In a relationship': 'Em um relacionamento',
        'Married': 'Casado',
        'Prefer not to say': 'Prefiro nao informar',
        'Single': 'Solteiro'
    },
    'cluster_comportamental': {
        'Binge Player': 'Jogador compulsivo',
        'Casual Enjoyer': 'Jogador casual',
        'Competitive Grinder': 'Competidor intenso',
        'Escape Seeker': 'Busca escape',
        'Social Gamer': 'Jogador social',
        'Streamer/Creator': 'Streamer/Criador',
        'Toxic Competitor': 'Competidor toxico'
    },
    'severidade_dependencia': {
        'None': 'Sem severidade',
        'Mild': 'Leve',
        'Moderate': 'Moderada',
        'Severe': 'Severa'
    },
    'severidade_dependencia_dashboard': {
        'None': 'Sem severidade',
        'Mild': 'Leve',
        'Moderate': 'Moderada',
        'Severe': 'Severa'
    }
}

for coluna, mapa in traducoes_valores.items():
    if coluna in df_powerbi.columns:
        df_powerbi[coluna] = df_powerbi[coluna].replace(mapa)

preenchimentos_categoricos = {
    'status_assinatura': 'Sem assinatura',
    'severidade_dependencia': 'Sem severidade',
    'severidade_dependencia_dashboard': 'Sem severidade'
}

for coluna, valor in preenchimentos_categoricos.items():
    if coluna in df_powerbi.columns:
        df_powerbi[coluna] = df_powerbi[coluna].fillna(valor).replace('', valor)

df_powerbi.head()

,id_usuario,idade,genero,pais,ocupacao,nivel_renda,anos_jogando,genero_preferido,plataforma,tipo_dispositivo,...,dias_absenteismo,velocidade_internet_mbps,horas_tela_total,cluster_comportamental,score_dependencia,dependencia_binaria,severidade_dependencia,probabilidade_burnout,score_risco_saude_mental,probabilidade_churn
0,USR000001,21,Masculino,Índia,Empregado,Média,9,Sandbox,PC,Notebook,...,7,39.9,5.4,Jogador casual,27.61,0,Leve,1.0,0.920,1.000
1,USR000002,25,Masculino,Índia,Empregado,Média-baixa,13,RPG,Mobile,Misto,...,6,71.5,13.4,Streamer/Criador,55.51,1,Moderada,1.0,0.515,0.813
2,USR000003,26,Masculino,Brasil,Empregado,Média,14,RPG,PC e mobile,PC high-end,...,7,119.4,12.3,Streamer/Criador,45.85,0,Moderada,1.0,0.720,0.947
3,USR000004,22,Masculino,Coréia do Sul,Empregado,Média-alta,10,Estratégia,PC e mobile,Mobile,...,8,136.5,6.9,Competidor toxico,39.87,0,Leve,1.0,0.520,0.660
4,USR000005,17,Feminino,Índia,Estudante,Média,5,Estratégia,PC,Notebook,...,6,78.8,9.3,Competidor intenso,46.97,0,Moderada,1.0,0.585,0.867


In [17]:
float_cols = df_powerbi.select_dtypes(include='float').columns

for col in float_cols:
   df_powerbi[col] = df_powerbi[col].map(
      lambda x: str(x).replace('.', ',') if pd.notna(x) else ''
   )

In [18]:
df_powerbi.head()

,id_usuario,idade,genero,pais,ocupacao,nivel_renda,anos_jogando,genero_preferido,plataforma,tipo_dispositivo,...,dias_absenteismo,velocidade_internet_mbps,horas_tela_total,cluster_comportamental,score_dependencia,dependencia_binaria,severidade_dependencia,probabilidade_burnout,score_risco_saude_mental,probabilidade_churn
0,USR000001,21,Masculino,Índia,Empregado,Média,9,Sandbox,PC,Notebook,...,7,"39,9","5,4",Jogador casual,"27,61",0,Leve,"1,0","0,92","1,0"
1,USR000002,25,Masculino,Índia,Empregado,Média-baixa,13,RPG,Mobile,Misto,...,6,"71,5","13,4",Streamer/Criador,"55,51",1,Moderada,"1,0","0,515","0,813"
2,USR000003,26,Masculino,Brasil,Empregado,Média,14,RPG,PC e mobile,PC high-end,...,7,"119,4","12,3",Streamer/Criador,"45,85",0,Moderada,"1,0","0,72","0,947"
3,USR000004,22,Masculino,Coréia do Sul,Empregado,Média-alta,10,Estratégia,PC e mobile,Mobile,...,8,"136,5","6,9",Competidor toxico,"39,87",0,Leve,"1,0","0,52","0,66"
4,USR000005,17,Feminino,Índia,Estudante,Média,5,Estratégia,PC,Notebook,...,6,"78,8","9,3",Competidor intenso,"46,97",0,Moderada,"1,0","0,585","0,867"


In [19]:
df_powerbi.to_csv('../data/processed/gaming_addiction_preprocessed.csv', index=False)

In [20]:
df_powerbi.to_excel('../data/processed/gaming_addiction_preprocessed_powerbi.xlsx', index=False)